# 01 — Representasi Citra Digital: dari Cahaya ke Data
Sebuah angka pada matriks citra bukan secara otomatis *intensitas cahaya* di permukaan objek. Angka itu merupakan hasil rangkaian proses: interaksi cahaya dengan benda, pembentukan bayangan oleh sistem optik, integrasi oleh sensor, konversi foton ke muatan listrik, pembacaan, penguatan, digitalisasi, dan sering kali pengolahan oleh kamera. Rangkaian inilah yang menentukan informasi apa yang dipertahankan—dan apa yang tidak mungkin dikembalikan lagi.

Notebook ini mengembangkan konsep representasi citra dari prinsip fisika dan matematika, lalu mengujinya melalui eksperimen terkontrol. **Seluruh percobaan dapat dijalankan tanpa dataset eksternal atau GPU.**

> **Pertanyaan pengantar:** Dua citra dengan ukuran array dan jumlah bit yang sama dapat mempunyai kualitas pengukuran yang berbeda jauh. Mengapa? Apakah menambah jumlah piksel selalu menambah informasi? Apakah `RGB=(128,128,128)` menyatakan setengah energi cahaya dari putih?

**Alur pembelajaran:** makna piksel → model sensor → sampling & aliasing → integrasi aperture/optik → kuantisasi → noise dan karakterisasi → warna dan encoding → kalibrasi spasial → konsekuensi terhadap machine learning → evaluasi kritis.

## Capaian pembelajaran dan prasyarat

Setelah eksperimen ini, pembaca diharapkan mampu:

1. Menurunkan model diskret pembentukan citra dari besaran radiometrik hingga nilai digital yang direkam.
2. Menjelaskan perbedaan sampel titik, rerata area piksel, interpolasi, dan representasi spasial yang terkalibrasi.
3. Memprediksi frekuensi alias, memeriksa batas Nyquist, dan memisahkan peran *pre-filter* dengan rekonstruksi.
4. Menghitung galat kuantisasi dan mengevaluasi syarat keberlakuan pendekatan varians $\Delta^2/12$.
5. Menghubungkan noise Poisson, noise pembacaan, penguatan sensor, serta keterbatasan *photon transfer curve*.
6. Membedakan data RGB terkode sRGB dari besaran warna linear dan luminans relatif.
7. Menilai perubahan representasi sebagai salah satu sumber *domain shift* pada model machine learning.
8. Menyatakan secara eksplisit asumsi, batas validitas, dan kesimpulan yang dapat didukung hasil eksperimen.

**Prasyarat:** dasar aljabar linear, kalkulus integral, probabilitas elementer, transformasi Fourier dasar, serta pemrograman Python/NumPy. Persamaan dikembangkan secara bertahap dan disertai penjelasan.

**Perkiraan aktivitas:** 2–3 pertemuan bila seluruh pertanyaan, derivasi, dan studi kasus dikerjakan; tidak semua eksperimen harus selesai dalam satu sesi.

## 1. Apa yang sebenarnya disimpan oleh sebuah citra?

Citra digital biasanya direpresentasikan oleh array $I\in\mathbb{R}^{H\times W}$ untuk satu kanal, atau $I\in\mathbb{R}^{H\times W\times C}$ untuk $C$ kanal. Indeks array $I[r,c]$ menyatakan posisi **baris** $r$ dan **kolom** $c$, bukan otomatis koordinat geometri $(x,y)$ suatu objek. Konvensi array: $r$ bertambah ke bawah, $c$ bertambah ke kanan. Konversi dari indeks ke posisi fisik memerlukan ukuran piksel, orientasi, model proyeksi, dan kalibrasi.

| Istilah | Makna | Pertanyaan untuk ilmuwan/insinyur |
|---|---|---|
| Nilai piksel | Angka keluaran suatu pipeline akuisisi | Apakah linear terhadap jumlah foton atau sudah melalui gamma/tone mapping? |
| Resolusi spasial | Kemampuan memisahkan struktur | Ditentukan oleh apa selain ukuran array? |
| *Bit depth* | Jumlah kemungkinan level representasi | Apakah level tambahan lebih kecil dari noise instrumen? |
| Rentang dinamis | Kisaran sinyal terukur sebelum noise/kliping mendominasi | Apakah rentang itu ditentukan ADC atau sensor? |
| Resolusi radiometrik | Kehalusan perbedaan sinyal yang dapat dibedakan | Apakah kuantisasi lebih dominan daripada noise? |
| *Ground truth* | Acuan untuk membandingkan hasil | Apakah berasal dari pengukuran, asumsi, anotasi, atau simulasi? |

**Pemisahan penting:** jumlah piksel $H\times W$ tidak identik dengan resolusi optik, nilai 8-bit tidak otomatis berarti akurasi radiometrik 8-bit, dan foto sRGB tidak otomatis dapat dipakai sebagai hasil radiometri.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage, stats
from skimage import data, transform, metrics
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

RNG = np.random.default_rng(2026)
plt.rcParams.update({'figure.dpi': 105, 'axes.grid': False, 'font.size': 10})

camera = data.camera().astype(np.float64) / 255.0
astronaut = data.astronaut().astype(np.float64) / 255.0
assert camera.ndim == 2 and astronaut.shape[-1] == 3

print('Python:', sys.version.split()[0])
print('Grayscale:', camera.shape, camera.dtype, 'rentang:', (camera.min(), camera.max()))
print('RGB:', astronaut.shape, astronaut.dtype, 'rentang:', (astronaut.min(), astronaut.max()))
print('Sumber: skimage.data.camera() dan skimage.data.astronaut() (citra contoh bawaan library).')

### 1.1 Memeriksa geometri array, bukan hanya menampilkan foto

Citra berikut berukuran $512\times512$ dengan nilai *float* ternormalisasi $[0,1]$. Normalisasi di sini **hanya penskalaan numerik dari citra referensi 8-bit**; tidak membuatnya berubah menjadi pengukuran radiometrik terkalibrasi. Sekalipun array diubah menjadi `float64`, informasi asli tidak bertambah.

Pilih area kecil untuk melihat makna koordinat baris–kolom dan melihat bahwa satu piksel merepresentasikan sel pengukuran, bukan titik tanpa luas.

In [ ]:
r0, c0, sisi = 180, 220, 9
potongan = camera[r0:r0+sisi, c0:c0+sisi]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), layout='constrained')
axes[0].imshow(camera, cmap='gray', vmin=0, vmax=1)
axes[0].plot([c0, c0+sisi, c0+sisi, c0, c0], [r0, r0, r0+sisi, r0+sisi, r0], 'r-', lw=1.6)
axes[0].set_title('Koordinat baris–kolom dan region of interest')
axes[0].set_xlabel('kolom c [piksel]'); axes[0].set_ylabel('baris r [piksel]')
im = axes[1].imshow(potongan, cmap='gray', vmin=0, vmax=1, interpolation='nearest')
axes[1].set_xticks(range(sisi), labels=np.arange(c0,c0+sisi))
axes[1].set_yticks(range(sisi), labels=np.arange(r0,r0+sisi))
axes[1].set_xlabel('kolom c'); axes[1].set_ylabel('baris r')
axes[1].set_title('Satu elemen array ≠ satu titik fisik')
plt.colorbar(im, ax=axes[1], label='nilai grayscale ternormalisasi')
plt.show()
print('Cuplikan numerik 3 × 3:')
print(np.round(potongan[:3,:3], 3))

## 2. Model pembentukan citra: dari radiansi ke sinyal digital

Untuk memperlihatkan apa yang disederhanakan oleh array citra, perhatikan medan radiansi spektral pada bidang citra $L_\lambda(x,y,t)$ dengan satuan contoh $\mathrm{W\,m^{-2}\,sr^{-1}\,nm^{-1}}$. Dalam sistem kamera umum, hubungan adegan, optik, dan sensor mencakup proyeksi geometri, throughput/aperture optik, respons spektral, difraksi, defokus, aberrasi, gerak, integrasi piksel, dan waktu paparan.

Sebuah model tereduksi untuk **rerata elektron** pada piksel $(r,c)$ dapat dituliskan

$$
\mu_{r,c} = \int_{t_0}^{t_0+T} \int_{\lambda} \int_{A_{r,c}}
\alpha(\lambda)\,[h_{\lambda,t}*L_{\lambda,t}](x,y)\,\mathrm{d}A\,\mathrm{d}\lambda\,\mathrm{d}t.
$$

Di sini:
- $T$ adalah waktu paparan [s]; $A_{r,c}$ adalah luas efektif piksel pada bidang sensor [m²].
- $h_{\lambda,t}$ merupakan *point spread function* (PSF) optik, ternormalisasi untuk bagian blur yang diasumsikan melestarikan energi; $*$ berarti konvolusi pada koordinat spasial.
- $\alpha(\lambda)$ merangkum konversi dari radiansi setelah geometri/optik menjadi laju elektron, termasuk faktor $\lambda/(hc)$, *quantum efficiency*, serta throughput. **Dimensi fisiknya harus ditetapkan ketika model dipakai untuk instrumen tertentu.**
- $\mu_{r,c}$ bersatuan elektron (angka ekspektasi), bukan nilai intensitas RGB.

Kemudian, untuk sensor yang mendekati linear sebelum pengolahan citra:

$$
N_{r,c}\sim\operatorname{Poisson}(\mu_{r,c}),\quad
Z_{r,c}=g\big(N_{r,c}+\epsilon_{\mathrm{read},r,c}\big)+o,\quad
D_{r,c}=Q_b\!\left(\operatorname{clip}(Z_{r,c},0,2^b-1)\right).
$$

- $N$ adalah elektron hasil deteksi, $\epsilon_{\rm read}$ noise pembacaan dalam elektron (di sini didekati Gaussian bermean nol), $g$ penguatan [DN/elektron], $o$ offset [DN], dan $D$ *digital number* (DN) keluaran ADC.
- $Q_b$ membulatkan ke kode ADC yang diizinkan; *clipping* menghilangkan informasi secara tak dapat balik.
- *Dark current*, *photo-response non-uniformity*, *fixed-pattern noise*, koreksi black-level, *demosaicing*, white balance, sharpening, kompresi, dan proses nonlinear **tidak dimodelkan lengkap di atas**. Banyak citra JPEG/PNG dari kamera tidak memenuhi model linear itu setelah *image signal processing*.

**Makna invers:** membaca $D_{r,c}$ tidak cukup untuk menentukan $L_\lambda(x,y,t)$ secara unik. Banyak adegan dan kondisi akuisisi berbeda dapat menghasilkan angka yang sama. Persoalan identifikasi tersebut muncul lagi pada notebook *inverse problems*.

### 2.1 Mengapa sinyal optik terhubung dengan teori sistem?

Jika respons spasial kamera dianggap linear dan *shift invariant* pada kondisi tertentu, citra sebelum sampling dapat ditulis sebagai

$$
g(x,y)=(f*h)(x,y)+\eta(x,y),
$$

lalu diintegrasikan pada area piksel dan didigitalkan. Dengan transformasi Fourier dua dimensi berlaku

$$
G(f_x,f_y)=F(f_x,f_y)H(f_x,f_y)+\mathcal{N}(f_x,f_y).
$$

$H$ adalah *optical transfer function* (OTF), dan $|H|$ yang dinormalisasi sering disebut *modulation transfer function* (MTF). Pernyataan ini membutuhkan asumsi sistem linear, invarian spasial, dan definisi noise yang konsisten. Keadaan umum—kamera nonlinier, saturasi, objek bergerak—tidak otomatis mengikuti persamaan tersebut.

**Implikasi:** frekuensi tinggi yang telah ditekan kuat oleh optik tidak bisa dipulihkan hanya dengan menaikkan ukuran array output. Sebaliknya, memperkecil piksel tanpa membatasi frekuensi optik dapat menghasilkan aliasing.

## 3. Sampling spasial dan batas Nyquist

Untuk kasus satu dimensi, misalkan pola kontinu $f(x)=a+b\cos(2\pi \nu x+\phi)$ [sinyal ternormalisasi], dan sampel diambil setiap $p$ meter:

$$
f[n]=a+b\cos(2\pi\nu np+\phi).
$$

Frekuensi sampling $\nu_s=1/p$ [siklus/m], sehingga batas Nyquist ialah $\nu_N=\nu_s/2=1/(2p)$. Bila $\nu>\nu_N$, terdapat frekuensi berbeda yang memberi urutan sampel identik. Secara sederhana, frekuensi terlipat adalah

$$
\nu_{\rm alias}=|\nu-k\nu_s|,
$$

untuk bilangan bulat $k$ yang membuat hasil berada dalam pita Nyquist. Ini tidak berarti pola kontinu *berubah frekuensi*; yang terjadi adalah **sampel tidak dapat membedakan pola yang berlainan**.

Teorema sampling rekonstruksi ideal mensyaratkan sinyal *band-limited*, sampling seragam, serta filter rekonstruksi ideal. Citra nyata memiliki tepi tajam, noise, dan spektrum yang tidak selalu terbatas secara ketat. Dalam dua dimensi, syarat Nyquist harus diperiksa terhadap kisi frekuensi dua dimensi, bukan hanya sepanjang arah horizontal.

In [ ]:
# Demonstrasi terkontrol: frekuensi 42 siklus pada panjang bidang yang sama.
N_high, faktor = 512, 8
N_low = N_high // faktor
n_h = np.arange(N_high)
frekuensi_siklus_bidang = 42
kontinu_didiskret = np.sin(2 * np.pi * frekuensi_siklus_bidang * n_h / N_high)
subsample = kontinu_didiskret[::faktor]
# Sinyal 42 siklus/64 sampel terlipat menjadi 22 siklus/64 sampel.
nu_alias = abs(frekuensi_siklus_bidang - N_low)
rekonstruksi_alias = np.sin(2*np.pi*(frekuensi_siklus_bidang/N_low)*np.arange(N_low))
assert np.allclose(subsample, rekonstruksi_alias)

fig, axes = plt.subplots(2, 1, figsize=(11, 6), layout='constrained')
axes[0].plot(n_h, kontinu_didiskret, lw=1, label='pola pada grid halus')
axes[0].set_xlim(0,128); axes[0].set_ylabel('amplitudo relatif')
axes[0].set_title('42 siklus per lebar bidang: informasi frekuensi sebelum penjarangan')
axes[0].legend()
axes[1].stem(np.arange(N_low), subsample, basefmt=' ', linefmt='C1-', markerfmt='C1o', label='hasil sampling')
axes[1].plot(np.arange(N_low), np.sin(2*np.pi*(-nu_alias)*np.arange(N_low)/N_low),
             'k--', alpha=.7, label='frekuensi bertanda −22 siklus/64')
axes[1].set_xlim(0,40); axes[1].set_xlabel('indeks sampel pada grid kasar')
axes[1].set_ylabel('amplitudo relatif'); axes[1].legend()
axes[1].set_title(f'Nyquist = {N_low/2:.0f} siklus/bidang; alias bermagnitudo {nu_alias}')
plt.show()
print('Frekuensi sumber:', frekuensi_siklus_bidang, 'siklus/bidang')
print('Nyquist grid kasar:', N_low/2, 'siklus/bidang')
print('Frekuensi alias:', nu_alias, 'siklus/bidang')

### 3.1 Sampling titik versus integrasi luas piksel

Model sensor ideal berupa sampel titik sering menjadi pendekatan pengajaran, tetapi fotodioda mempunyai luasan. Untuk aperture piksel persegi yang merata, pengukuran rata-rata area pada satu sel

$$
\bar f_{r,c}=\frac{1}{A_{r,c}}\int_{A_{r,c}} f(x,y)\,\mathrm{d}A
$$

menekan variasi yang sangat halus sebelum data diperkecil. Ini sejenis *low-pass filtering* yang terjadi melalui integrasi spasial, meskipun respons nyata juga dipengaruhi optik dan *fill factor*. Mengambil setiap piksel kedelapan dengan `I[::8,::8]` adalah **decimation tanpa pre-filter**, bukan model lengkap pembacaan sensor dengan piksel delapan kali lebih besar.

Berikut tiga cara menghasilkan gambar resolusi lebih rendah. Karena hasil eksperimen adalah simulasi komputasional dari gambar referensi, kita tidak menyebutnya validasi sensor fisik.

In [ ]:
faktor = 8
potong = camera[::faktor, ::faktor]
anti_alias = transform.resize(camera, potong.shape, order=1, anti_aliasing=True,
                             preserve_range=True)
rerata_area = transform.downscale_local_mean(camera, (faktor, faktor))

fig, axes = plt.subplots(1, 3, figsize=(12, 4), layout='constrained')
for ax, x, nama in zip(axes, [potong, anti_alias, rerata_area],
                       ['Subsample tanpa pre-filter', 'Filter + downsample', 'Rerata area 8×8']):
    ax.imshow(x, cmap='gray', vmin=0, vmax=1, interpolation='nearest')
    ax.set_title(nama); ax.set_xlabel('kolom'); ax.set_ylabel('baris')
plt.show()

def rmse(a,b):
    a=np.asarray(a,dtype=float); b=np.asarray(b,dtype=float)
    return float(np.sqrt(np.mean((a-b)**2)))

# Target operasionalnya adalah rerata area, bukan 'kebenaran optik' dunia nyata.
print('RMSE terhadap representasi rerata area 8×8:')
print('  tanpa pre-filter:', round(rmse(potong,rerata_area), 4))
print('  Gaussian anti-alias + resize:', round(rmse(anti_alias,rerata_area), 4))
print('Catatan: rerata area dipilih sebagai definisi target pada percobaan ini, bukan sebagai ground truth sensor.')

### 3.2 Eksperimen aliasing dua dimensi dengan pola buatan

Pola sinusoidal dua dimensi di bawah memiliki frekuensi yang **diketahui**. Kita akan membandingkan subsampling dengan integrasi area. Citra ini **sintetis**, sehingga angka siklusnya terkontrol; hasil tidak boleh diinterpretasikan sebagai pengujian resolusi kamera sungguhan.

In [ ]:
N=384; dec=6
xx, yy=np.meshgrid(np.arange(N),np.arange(N))
# Frekuensi setelah decimation: (43/64, 37/64) siklus/piksel -> > Nyquist.
pola=0.5+0.22*np.sin(2*np.pi*(43*xx+37*yy)/N)
langsung=pola[::dec,::dec]
lowpass=transform.downscale_local_mean(pola, (dec,dec))
fig,axes=plt.subplots(1,3,figsize=(12,4),layout='constrained')
for ax,z,j in zip(axes,[pola,langsung,lowpass],
                   ['Pola resolusi tinggi', 'Decimation: alias', 'Integrasi area (pre-filter)']):
    ax.imshow(z,cmap='gray',vmin=0,vmax=1,interpolation='nearest')
    ax.set_title(j);ax.set_xticks([]);ax.set_yticks([])
plt.show()
print('Standar deviasi pada grid kasar:',
      'subsample =',round(float(langsung.std()),4),
      '| integrasi area =',round(float(lowpass.std()),4))

### 3.3 Hubungan resolusi spasial dan MTF

Untuk PSF Gaussian ideal, $h(x,y)\propto\exp[-(x^2+y^2)/(2\sigma^2)]$, respons magnitudo terhadap frekuensi spasial radial $\rho$ (dalam siklus per piksel pada grid model) adalah

$$
\operatorname{MTF}(\rho)=\exp(-2\pi^2\sigma^2\rho^2).
$$

Fungsi tersebut menunjukkan redaman yang meningkat pada frekuensi tinggi. Formula ini berlaku untuk PSF Gaussian yang dinormalisasi—bukan model universal semua lensa. Nyquist adalah sifat sampling; MTF adalah sifat transfer frekuensi sistem optik/sensor. Keduanya berinteraksi tetapi **tidak dapat dipertukarkan**.

In [ ]:
f=np.linspace(0,.5,300)
sigma_list=[.4,.8,1.5,2.5]
fig,axes=plt.subplots(1,2,figsize=(11,4),layout='constrained')
for sig in sigma_list:
    axes[0].plot(f,np.exp(-2*np.pi**2*sig**2*f**2),label=f'σ = {sig} px')
axes[0].axvline(.5,color='k',ls=':',label='Nyquist (grid ini)')
axes[0].set(xlabel='frekuensi spasial [siklus/piksel]',ylabel='MTF relatif',
            title='Gaussian PSF: pelemahan komponen frekuensi')
axes[0].legend()
blur=ndimage.gaussian_filter(camera,sigma=2.0,mode='reflect')
axes[1].plot(camera[230,:],alpha=.75,label='profil asli')
axes[1].plot(blur[230,:],alpha=.9,label='setelah blur Gaussian σ=2 px')
axes[1].set(xlabel='kolom [piksel]',ylabel='nilai grayscale relatif',
            title='Contoh kehilangan detail pada profil intensitas')
axes[1].legend()
plt.show()

## 4. Kuantisasi: mengubah besaran kontinu menjadi sejumlah kode

Jika nilai sinyal $x$ dibatasi pada interval $[0,1]$, salah satu kuantisator titik ujung memiliki $L=2^b$ level dan langkah

$$
\Delta=\frac{1}{L-1},\qquad Q_b(x)=\frac{\operatorname{round}[(L-1)\operatorname{clip}(x,0,1)]}{L-1}.
$$

Dengan pembulatan ke level terdekat, untuk nilai masukan **yang tidak terklip** berlaku $|Q_b(x)-x|\leq\Delta/2$. Rumus $\Delta=1/(L-1)$ spesifik untuk konvensi kuantisator titik ujung ini, sedangkan ADC fisik dapat memakai kode interval dan aturan saturasi berbeda.

**Model galat klasik:** apabila galat kuantisasi dapat dianggap tersebar seragam pada $[-\Delta/2,\Delta/2]$, tidak berkorelasi dengan sinyal, dan tidak ada kliping, maka

$$
\mathbb{E}[e]\approx0,\qquad \operatorname{Var}(e)\approx\frac{\Delta^2}{12}.
$$

Pendekatan ini **bukan identitas yang berlaku untuk semua citra**. Pada bidang hampir konstan, pola periodik, level sangat sedikit, atau sinyal yang terkunci pada level tertentu, galat bisa sangat berkorelasi dengan sinyal dan mempunyai bias.

In [ ]:
def kuantisasi_titik_ujung(x,bits):
    if not isinstance(bits,int) or bits<1: raise ValueError('bits harus bilangan bulat positif')
    L=2**bits
    return np.rint(np.clip(np.asarray(x,dtype=float),0,1)*(L-1))/(L-1)

bits_daftar=[1,2,4,8]
fig,axes=plt.subplots(1,4,figsize=(12,3.4),layout='constrained')
records=[]
for ax,b in zip(axes,bits_daftar):
    out=kuantisasi_titik_ujung(camera,b)
    err=rmse(camera,out)
    records.append({'bit':b,'level':2**b,'delta':1/(2**b-1),'RMSE':err})
    ax.imshow(out,cmap='gray',vmin=0,vmax=1)
    ax.set_title(f'{b} bit | RMSE={err:.4f}')
    ax.set_xticks([]);ax.set_yticks([])
plt.show()
print(pd.DataFrame(records).to_string(index=False, float_format=lambda a:f'{a:.6f}'))
print('Sumber camera adalah citra 8-bit, sehingga 8-bit endpoint memberi galat ≈ 0 terhadap data referensi.')

### 4.1 Menguji kapan varians galat mendekati $\Delta^2/12$

Tidak cukup hanya mencocokkan grafik dengan sebuah rumus. Kita harus memeriksa asumsi distribusi masukan dan lokasi nilai dalam sel kuantisasi. Pada sampel ramp yang hampir seragam dan cukup halus, distribusi residu cenderung mendekati bentuk seragam. Untuk data hampir konstan yang persis dekat satu level, galatnya tidak lagi bersifat acak terhadap masukan.

In [ ]:
b=4;delta=1/(2**b-1)
ramp=np.linspace(0,1,200_001)
err_ramp=kuantisasi_titik_ujung(ramp,b)-ramp
konstan=np.full_like(ramp,.21)
err_konstan=kuantisasi_titik_ujung(konstan,b)-konstan
fig,axes=plt.subplots(1,2,figsize=(11,3.7),layout='constrained')
axes[0].hist(err_ramp,bins=45,density=True,alpha=.75)
axes[0].axvline(-delta/2,c='k',ls=':');axes[0].axvline(delta/2,c='k',ls=':')
axes[0].set(xlabel='galat kuantisasi',ylabel='densitas',title='Galat pada ramp: mendekati sebaran seragam')
axes[1].plot(ramp[::300],err_ramp[::300],'.',markersize=1.8,label='ramp')
axes[1].axhline(err_konstan[0],c='C3',label='sinyal konstan 0.21')
axes[1].set(xlabel='intensitas masukan',ylabel='galat',title='Galat dapat deterministik dan bergantung masukan')
axes[1].legend()
plt.show()
print('Varians ramp numerik  :', f'{np.var(err_ramp):.7f}')
print('Pendekatan Δ² / 12   :', f'{delta**2/12:.7f}')
print('Varians sinyal konstan:', f'{np.var(err_konstan):.7f}', '| bias:',f'{np.mean(err_konstan):.5f}')

### 4.2 Kuantisasi, saturasi, dan rentang dinamis merupakan persoalan berbeda

*Bit depth* menentukan jumlah kode, tetapi rentang dinamis praktis ditentukan oleh rasio sinyal terbesar sebelum saturasi terhadap noise terendah yang masih relevan bagi pengukuran. Sensor 16-bit dengan noise besar dan rentang tegangan terbatas tidak otomatis mengukur lebih akurat dibanding sensor 12-bit yang terkalibrasi dengan baik. Penambahan bit pada data yang sudah terklip **tidak dapat mengembalikan** sinyal yang hilang.

Eksperimen berikut secara sengaja membuat sinyal melewati batas nilai yang dapat direpresentasikan.

In [ ]:
x=np.linspace(0,1.5,1000)
y=kuantisasi_titik_ujung(x,4)
fig,ax=plt.subplots(figsize=(9,3.5),layout='constrained')
ax.plot(x,x,label='masukan (sebelum sensor)',alpha=.7)
ax.step(x,y,where='mid',label='ADC 4-bit + saturasi',lw=2)
ax.axvline(1.,color='C3',ls='--',label='awal saturasi')
ax.set(xlabel='sinyal masuk (normalisasi kapasitas penuh)',ylabel='keluaran terkode',
       title='Banyak nilai masukan → kode maksimum yang sama')
ax.legend();plt.show()
assert np.all(kuantisasi_titik_ujung(np.array([1.1,1.5]),4)==1)

## 5. Noise sensor dan *photon transfer curve*

Pada intensitas rendah, jumlah elektron yang terdeteksi acak karena proses kedatangan foton. Untuk $N\sim\operatorname{Poisson}(\mu)$ berlaku $\mathbb{E}[N]=\mu$ dan $\operatorname{Var}(N)=\mu$. Bila noise pembacaan memiliki varians $\sigma_r^2$ dalam elektron, maka sebelum kliping dan asumsi noise independen:

$$
\operatorname{Var}(Z)=g^2(\mu+\sigma_r^2).
$$

Bila kuantisasi dapat dimodelkan sebagai noise seragam independen dengan langkah **1 DN**, diperoleh pendekatan

$$
\operatorname{Var}(D)\approx g^2(\mu+\sigma_r^2)+\frac{1}{12}.
$$

Nilai rerata sinyal digital tanpa offset adalah $S=\mathbb{E}[D]-o\approx g\mu$. Eliminasi $\mu$ menghasilkan

$$
\operatorname{Var}(D)\approx gS+g^2\sigma_r^2+\frac{1}{12}.
$$

Hubungan linear antara varians temporal dan mean sinyal memungkinkan estimasi *conversion gain* lewat kemiringan pada model ideal ini. Model di atas mengabaikan dark current, PRNU, pixel nonlinearity, clipping, serta efek pemrosesan kamera. **Varians antar-piksel satu gambar tidak sama dengan varians temporal dari pengukuran berulang**; struktur objek dan pola tetap sensor dapat mengacaukannya.

In [ ]:
# Simulasi sensor yang linear pada domain elektron/DN; bukan data sensor laboratorium.
mu_daftar=np.array([12,30,70,130,260,500,850],dtype=float)
g=0.55                  # DN per elektron
sigma_read=4.0          # elektron RMS
offset=90.0             # DN
n_frame=28              # repetisi paparan per level
ukuran=(48,48)          # area flat-field sintetis
hasil=[]
for mu in mu_daftar:
    elektron=RNG.poisson(mu,size=(n_frame,*ukuran))
    pembacaan=RNG.normal(0,sigma_read,size=(n_frame,*ukuran))
    dn=np.rint(np.clip(g*(elektron+pembacaan)+offset,0,4095))
    rerata_pixel=dn.mean(axis=0)
    var_temporal_pixel=dn.var(axis=0,ddof=1)
    hasil.append({'mu_e':mu,'rerata_DN':dn.mean(),
                  'sinyal_DN':dn.mean()-offset,
                  'varians_temporal_DN2':var_temporal_pixel.mean()})
hasil=pd.DataFrame(hasil)
# Regresi varians vs signal = gain * signal + intercept.
fit=np.polyfit(hasil['sinyal_DN'],hasil['varians_temporal_DN2'],1)
fig,ax=plt.subplots(figsize=(8.6,4.4),layout='constrained')
ax.scatter(hasil['sinyal_DN'],hasil['varians_temporal_DN2'],label='simulasi flat-field')
xfit=np.linspace(0,hasil['sinyal_DN'].max(),200)
ax.plot(xfit,np.polyval(fit,xfit),label='regresi empiris',color='C1')
ax.plot(xfit,g*xfit+g*g*sigma_read**2+1/12,'k:',label='teori (asumsi ideal)')
ax.set(xlabel='rata-rata DN − offset [DN]',ylabel='varians temporal [DN²]',
       title='Photon transfer: ulangan pengukuran per level paparan')
ax.legend();plt.show()
print(hasil.round(3).to_string(index=False))
print(f'Gain nominal = {g:.3f} DN/e; estimasi kemiringan = {fit[0]:.3f} DN/e')
print(f'Intersep empiris = {fit[1]:.3f} DN²; intersep teori ≈ {g*g*sigma_read**2+1/12:.3f} DN²')
assert abs(fit[0]-g)<0.12, 'Estimasi gain jauh dari simulasi; periksa implementasi!'

### 5.1 Noise tidak selalu terlihat sama pada semua level sinyal

*Shot noise* meningkat pada varians seiring $\mu$, tetapi rasio sinyal terhadap noise Poisson meningkat kira-kira seperti $\sqrt{\mu}$ bila shot noise dominan. Oleh sebab itu, sebuah foto dengan paparan lebih terang dapat memiliki simpangan baku absolut lebih besar sekaligus SNR lebih baik. Menyatakan "noise naik" tanpa menyebut *metrik* dapat menyesatkan.

$$
\mathrm{SNR}(\mu) = \frac{\mu}{\sqrt{\mu+\sigma_r^2}}\quad \text{(rasio amplitudo dalam satuan elektron; model sederhana).}
$$

Untuk analisis akurasi dan batas deteksi alat ukur, model noise harus diukur ulang pada instrumen dan protokol yang digunakan, bukan ditetapkan hanya karena grafik simulasi tampak masuk akal.

In [ ]:
mu=np.geomspace(1,2000,300)
snr=mu/np.sqrt(mu+sigma_read**2)
fig,ax=plt.subplots(figsize=(8.5,3.7),layout='constrained')
ax.loglog(mu,np.sqrt(mu+sigma_read**2),label='simpangan baku [elektron]')
ax.loglog(mu,snr,label='SNR amplitudo [tanpa satuan]')
ax.axvline(sigma_read**2,c='gray',ls=':',label='μ = σ_read²')
ax.set(xlabel='rerata elektron per piksel, μ [elektron]',title='Noise absolut dan SNR: tren yang berbeda')
ax.legend();plt.show()

## 6. Representasi warna: linearitas dan transfer function sRGB

Nilai RGB terkode bukan secara langsung tiga pengukuran energi spektral. Sensor warna memiliki respons spektral $s_k(\lambda)$, sehingga model sederhana kanal mentah adalah

$$
m_k=\int E(\lambda)R(\lambda)s_k(\lambda)\,\mathrm{d}\lambda+\epsilon_k,
$$

dengan $E(\lambda)$ distribusi iluminasi yang tiba pada objek (beserta faktor geometri), $R(\lambda)$ reflektansi spektral, dan $s_k(\lambda)$ sensitivitas spektral kanal ke-$k$. **Warna teramati dipengaruhi objek, illuminant, sensor, serta pipeline pemrosesan**; tidak ada pemetaan unik dari tiga nilai RGB ke spektrum reflektansi.

Untuk citra RGB yang memang terkode **sRGB**, kanal ternormalisasi $C_s\in[0,1]$ didekode menjadi kanal linear $C_l$:

$$
C_l=\begin{cases}
C_s/12.92,&C_s\le0.04045,\\
\left(\frac{C_s+0.055}{1.055}\right)^{2.4},&C_s>0.04045.
\end{cases}
$$

Dengan asumsi ruang warna sRGB dan *white point* standar, luminans **relatif** dari kanal linear diperoleh dari

$$
Y=0.2126R_l+0.7152G_l+0.0722B_l.
$$

Angka ini bukan otomatis luminans absolut dalam $\mathrm{cd/m^2}$; diperlukan kalibrasi perangkat/scene untuk itu. Menggunakan bobot pada kanal yang masih terkode menghasilkan *luma-like* (bukan $Y$ linear). Rerata aritmetika tiga kanal lebih jauh lagi tidak mempertahankan pembobotan perseptual atau fotometrik. Perbedaan koefisien kecil dalam beragam library dapat muncul dari standar/konvensi implementasi.

In [ ]:
def srgb_ke_linear(c):
    c=np.asarray(c,dtype=float)
    if np.any((c<0)|(c>1)): raise ValueError('Nilai sRGB harus dalam [0,1]')
    return np.where(c<=0.04045,c/12.92,((c+0.055)/1.055)**2.4)

def linear_ke_srgb(c):
    c=np.asarray(c,dtype=float)
    if np.any((c<0)|(c>1)): raise ValueError('Nilai linear harus dalam [0,1]')
    return np.where(c<=0.0031308,12.92*c,1.055*np.maximum(c,0)**(1/2.4)-0.055)

bobot=np.array([.2126,.7152,.0722])
Y_lin=(srgb_ke_linear(astronaut)*bobot).sum(axis=-1)
y_luma=(astronaut*bobot).sum(axis=-1)
y_rata=astronaut.mean(axis=-1)

fig,axes=plt.subplots(1,4,figsize=(14,4),layout='constrained')
for ax,arr,nama in zip(axes,[astronaut,Y_lin,y_luma,y_rata],
                       ['Warna (diasumsikan sRGB)','Luminans relatif linear Y',
                        'Bobot pada sRGB terkode','Rata-rata kanal RGB']):
    ax.imshow(arr,cmap='gray' if arr.ndim==2 else None,vmin=0,vmax=1)
    ax.set_title(nama,fontsize=9);ax.set_xticks([]);ax.set_yticks([])
plt.show()
print('Contoh abu-abu sRGB 0,5 → linear:', round(float(srgb_ke_linear(.5)),4))
print('RMS beda Y linear vs luma-terkode:',f'{rmse(Y_lin,y_luma):.4f}')
assert np.isclose(srgb_ke_linear(linear_ke_srgb(.18)),.18,atol=1e-10)

### 6.1 Mengapa gambar *terlihat* sama tetapi hasil pengukuran bisa berbeda?

Tampilan layar melewati kurva transfer, pengelolaan warna, dan kalibrasi monitor. Menyamakan hasil visual pada monitor dengan kesetaraan besaran fisik adalah kesalahan metodologis. Misalnya, nilai sRGB $0.5$ dalam satu kanal berkaitan dengan nilai linear sekitar $0.214$, bukan $0.5$.

Jika penelitian mengandalkan perubahan warna untuk mengukur konsentrasi zat atau mutu material, dokumentasikan minimal: illuminant, geometri pencahayaan, respons spektral/karakterisasi kamera, white balance, exposure, mode RAW/JPEG, koreksi warna, serta rentang nilai yang dipakai sebelum ekstraksi fitur.

## 7. Dtype, skala intensitas, dan perangkap representasi numerik

Banyak kesalahan tersembunyi bukan berasal dari algoritma citra, melainkan dari kesalahan konversi tipe dan skala. Di NumPy, `uint8` hanya menyimpan bilangan bulat tak bertanda 0–255. Operasi yang menyimpang dari rentang itu dapat mengalami overflow jika dijalankan pada dtype tetap; sementara pembagian dengan 255 menghasilkan angka pecahan tanpa menambah informasi asli. `scikit-image` menggunakan konvensi khusus untuk konversi dtype sehingga harus dipahami sebelum memakai `preserve_range`.

Periksa contoh berikut. Angka pada array yang sama dapat dipahami secara berbeda ketika metadata/konvensi tidak dicatat.

In [ ]:
piksel=np.array([0,10,127,245,255],dtype=np.uint8)
# Pastikan operasi dilakukan dalam bilangan bertipe lebih lebar sebelum clipping.
tambah_aman=np.clip(piksel.astype(np.int16)+25,0,255).astype(np.uint8)
tambah_modulo=(piksel+np.uint8(25))
print('Asal uint8       :',piksel)
print('Tambah 25 aman  :',tambah_aman)
print('Tambah 25 wrap  :',tambah_modulo)
print('Normalisasi float:', np.round(piksel.astype(float)/255,4))
assert tambah_modulo[-1] == 24

## 8. Kalibrasi spasial: dari piksel ke panjang fisik

Tanpa kalibrasi, panjang 10 piksel hanya berarti 10 elemen grid. Pada pencitraan planar sederhana dengan ukuran piksel efektif $p_x,p_y$ dan transformasi geometri yang diketahui, pusat elemen $(r,c)$ dapat dipetakan, misalnya, ke

$$
x_c=(c+1/2)p_x,\qquad y_r=(r+1/2)p_y.
$$

Konvensi setengah piksel di atas bergantung pada definisi tepi kisi dan origin; yang penting adalah **menetapkan satu konvensi yang konsisten**. Bila citra diperoleh dengan kamera perspektif, jarak pada sensor tidak otomatis sebanding dengan jarak pada objek tanpa pengetahuan magnifikasi, bidang objek, kalibrasi intrinsik/ekstrinsik, dan distorsi.

Dalam pengukuran luas objek dengan piksel anisotropik, $A\approx N_{\rm piksel}p_xp_y$; ketidakpastian batas segmentasi dapat mendominasi ketidakpastian kalibrasi. Untuk objek bundar berdiameter $d$ yang diperkirakan dari $d_{\rm px}$ piksel, $d=d_{\rm px}\,p$ dan propagasi galat pendekatan orde pertama (bila independen)

$$
\left(\frac{u_d}{d}\right)^2\approx\left(\frac{u_{d_{\rm px}}}{d_{\rm px}}\right)^2+\left(\frac{u_p}{p}\right)^2.
$$

Notasi $u$ menunjukkan *standard uncertainty*. Persamaan sederhana ini tidak berlaku begitu saja jika hasil pengukuran saling berkorelasi.

In [ ]:
p_um=3.45      # contoh ukuran efektif [mikrometer / pixel]; bukan metadata data.camera()
u_p_um=.03    # standard uncertainty kalibrasi, asumsi ilustratif
d_px=46.0     # pengukuran diameter, sintetis
u_d_px=1.0    # standard uncertainty penentuan tepi, asumsi ilustratif
d_um=d_px*p_um
u_d_um=np.sqrt((p_um*u_d_px)**2+(d_px*u_p_um)**2)
print(f'Diameter ilustratif = ({d_um:.2f} ± {u_d_um:.2f}) µm (standard uncertainty, k=1)')
print(f'Kontribusi relatif batas segmentasi: {u_d_px/d_px:.2%}')
print(f'Kontribusi relatif kalibrasi: {u_p_um/p_um:.2%}')
print('Keterangan: angka di atas adalah contoh perhitungan, BUKAN pengukuran objek nyata.')

## 9. Mini-studi kasus: pengaruh representasi terhadap machine learning

Di sini representasi citra dinilai berdasarkan konsekuensinya terhadap sebuah tugas hilir. Dataset **Digits** (8×8 piksel, label digit tulisan tangan) tersedia di scikit-learn. Agar tidak terjadi *data leakage*, data dibagi satu kali menjadi training dan test secara stratified, semua parameter `StandardScaler` dan SVM dipelajari **hanya pada training**, lalu test dibaca sebagai evaluasi tanpa tuning lanjutan.

Kita menguji beberapa perubahan **hanya pada test set** untuk mensimulasikan domain shift representasi: pengurangan bit dan transformasi gamma yang memengaruhi distribusi intensitas. Eksperimen ini tidak menyatakan bahwa satu metode kuantisasi pasti lebih buruk pada seluruh dataset. Perubahan test-only disengaja untuk menguji ketahanan terhadap akuisisi berbeda; bila ingin mengoptimalkan ketahanan model, gunakan validation set terpisah dan jangan menyesuaikan model setelah melihat test.

Dataset Digits tidak memiliki metadata sensor, noise fisik, atau illuminant. Karena itu, eksperimen ini adalah kajian numerik representasi, **bukan bukti domain shift kamera industri**.

In [ ]:
digits=load_digits()
X=np.clip(digits.data.astype(float)/16.0,0,1)
y=digits.target
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=2026,stratify=y)
model=make_pipeline(StandardScaler(),SVC(kernel='rbf',C=3.0,gamma='scale'))
model.fit(Xtr,ytr)

variasi={
    'Representasi asli':Xte,
    'Kuantisasi 4-bit':kuantisasi_titik_ujung(Xte,4),
    'Kuantisasi 2-bit':kuantisasi_titik_ujung(Xte,2),
    'Encoding nonlinier (x^0,6)':np.power(Xte,.6),
    'Pelemahan intensitas (0,65x)':.65*Xte,
}
hasil_ml=[]
for nama,z in variasi.items():
    akurasi=float(np.mean(model.predict(z)==yte))
    hasil_ml.append({'kondisi_test':nama,'akurasi':akurasi,'jumlah_sampel':len(yte)})
df_ml=pd.DataFrame(hasil_ml)
print(df_ml.to_string(index=False,float_format=lambda v:f'{v:.4f}'))
fig,axes=plt.subplots(1,2,figsize=(11,3.8),layout='constrained')
axes[0].imshow(Xte[3].reshape(8,8),cmap='gray',vmin=0,vmax=1,interpolation='nearest')
axes[0].set_title('Contoh data test asli');axes[0].axis('off')
axes[1].barh(df_ml['kondisi_test'],df_ml['akurasi'])
axes[1].invert_yaxis();axes[1].set_xlim(0,1)
axes[1].set_xlabel('akurasi pada test set tetap');axes[1].set_title('Perubahan representasi → perubahan performa')
plt.show()

### 9.1 Bagaimana menginterpretasikan studi kasus ML secara sahih?

- Perbedaan akurasi pada **satu split** adalah temuan eksploratif dan belum menjadi estimasi generalisasi dengan interval kepercayaan.
- Karena observasi yang sama diuji pada semua kondisi, analisis perbandingan lanjutan harus mempertimbangkan data berpasangan dan desain pengujian. Angka pada tabel bukan uji signifikansi.
- Sebagian distorsi mungkin secara kebetulan tidak menurunkan performa atau bahkan mengubahnya sedikit ke atas. Itu tidak membenarkan klaim bahwa distorsi tersebut meningkatkan kualitas citra.
- Kualitas citra merupakan konsep tugas-spesifik: citra yang tampak lebih tajam belum tentu menghasilkan pengukuran atau inferensi yang lebih akurat.
- Untuk riset nyata, diperlukan data dari instrumen, batch, dan lingkungan yang sesuai, beserta protokol validasi *out-of-domain*.

**Pertanyaan kritis:** Jika akurasi turun ketika intensitas berubah, apakah kita telah menunjukkan model tidak robust, atau hanya bahwa pra-pemrosesan dan distribusi training tidak kompatibel? Data tambahan apa yang diperlukan untuk membedakannya?

## 10. Sintesis: rantai sebab-akibat dan keterbatasan inferensi

Perhatikan hubungan berikut, yang akan diperdalam dalam notebook lanjutan:

| Keputusan dalam akuisisi/representasi | Kerugian atau distorsi informasi | Dampak potensial terhadap analisis |
|---|---|---|
| Optical blur / PSF | Redaman frekuensi spasial tinggi | Batas resolusi, ketidakpastian deteksi tepi |
| Sampling di bawah kebutuhan Nyquist | Aliasing yang tidak dapat dipisahkan dari sinyal sah | Pola semu dan bias pengukuran |
| Integrasi area piksel | Rerata lokal, hilangnya variasi mikro | Ketelitian sub-piksel berkurang, noise dapat berubah |
| Kuantisasi dengan sedikit level | Kesalahan numerik terstruktur/acak | Hilangnya detail kecil, perubahan fitur |
| Saturasi ADC | Banyak masukan → nilai sama | Informasi hilang permanen |
| Noise foton/pembacaan | Fluktuasi hasil pengukuran | Penurunan presisi, kebutuhan estimasi ketidakpastian |
| Encoding warna dan white balance | Pemetaan nonlinear dan penyesuaian kanal | Fitur warna menjadi bergantung pada pipeline |
| Transformasi pada dataset | Pergeseran distribusi | Penurunan atau perubahan performa model ML |

Tabel ini menguraikan *potensi*, bukan hasil yang berlaku tanpa syarat. Besar dampak harus dihitung menurut model alat, data, dan tugas tertentu.

## 11. Latihan berjenjang dan pembahasan berbasis bukti

**A. Pemahaman konsep**

1. Mengapa $I[r,c]=120$ tidak selalu berarti intensitas fisik yang sama pada dua kamera? Sebutkan sedikitnya lima komponen rantai akuisisi yang dapat mengubahnya.
2. Jelaskan perbedaan antara jumlah piksel, kemampuan resolusi optik, dan ketelitian ukuran benda dalam satu contoh pengukuran mikroskop.
3. Apakah `astype(float)/255` meningkatkan informasi suatu citra 8-bit? Apa keuntungan operasi itu dan apa yang tidak dapat diperolehnya?

**B. Analisis kuantitatif**

4. Sensor dengan jarak antar-pusat piksel $p=5\,\mu\mathrm{m}$ mengukur pola sinusoidal $\nu=140\,\mathrm{siklus/mm}$. Hitung $\nu_s$, $\nu_N$, dan frekuensi alias. Jelaskan mengapa filter anti-alias perlu bekerja **sebelum**, bukan sesudah sampling.
5. Pada kuantisator titik ujung 5-bit ternormalisasi, tentukan $L$, $\Delta$, galat absolut maksimum (tanpa kliping), dan pendekatan simpangan baku kuantisasi jika model noise seragam berlaku.
6. Model sensor mempunyai $g=0.5\,\mathrm{DN/e^-}$, $\sigma_r=3\,\mathrm{e^-}$, $\mu=100\,\mathrm{e^-}$ dan tidak mengalami kliping. Prediksi rerata sinyal relatif terhadap offset dan varians DN dengan asumsi kuantisasi seragam berlangkah 1 DN. Jelaskan tiga alasan mengapa pengukuran nyata mungkin berbeda.

**C. Eksperimen dan HOTS**

7. Kembangkan percobaan yang menunjukkan bahwa kenaikan ukuran citra melalui interpolasi tidak sama dengan peningkatan resolusi optik. Gunakan satu metrik dan satu interpretasi profil spasial; identifikasi syarat yang membuat metrik itu bermakna.
8. Pada sampel warna yang diukur dengan smartphone, pembacaan RGB berubah ketika pencahayaan ruangan bergeser. Rancang protokol untuk memisahkan pengaruh illumination, auto white balance, saturasi, dan perubahan sampel. Bagaimana menyatakan ketidakpastiannya?
9. Rancang perbandingan dua representasi input bagi model klasifikasi dengan *grouped split* menurut perangkat kamera. Uraikan cara mencegah data leakage dari objek yang difoto berulang, metode evaluasi, dan batas kesimpulan.

**Panduan pelaporan:** tulis asumsi → persamaan → perhitungan → visualisasi → interpretasi → potensi bias → batas klaim. Untuk tugas HOTS, sertakan setidaknya satu hipotesis alternatif yang juga dapat menjelaskan hasil.

## 12. Pemeriksaan mandiri: uji numerik dan reproducibility

Tes di bawah tidak membuktikan model eksperimen merupakan representasi sempurna kamera sungguhan. Namun, tes menilai beberapa invarian penting: batas galat kuantisasi, linearitas kasus sederhana, identitas transformasi warna, dan konsistensi numerik. Jika ada kegagalan, jangan menyembunyikannya dengan menghapus sel.

In [ ]:
# Tes numerik cepat: seluruhnya dapat berjalan tanpa akses jaringan.
x_uji=np.linspace(0,1,100_001)
for b in [1,2,4,6,8,12]:
    q=kuantisasi_titik_ujung(x_uji,b)
    assert np.max(np.abs(q-x_uji)) <= 0.5/(2**b-1)+1e-12
    assert np.all((q>=0)&(q<=1))

warna_linear=RNG.uniform(0,1,size=(100,3))
assert np.allclose(srgb_ke_linear(linear_ke_srgb(warna_linear)),warna_linear,atol=1e-12)
assert np.allclose(transform.downscale_local_mean(np.ones((24,24)), (4,4)),1)
assert camera.shape == (512,512)
assert len(yte)==len(model.predict(Xte))
print('LULUS — uji batas kuantisasi, invers transfer sRGB, konservasi citra konstan, dan pipeline ML.')

## 13. Kesimpulan ilmiah dan batas klaim

1. Representasi digital adalah hasil operator pembentukan citra, interaksi noise, dan pengkodean, sehingga interpretasi numeriknya bergantung pada proses akuisisi.
2. Sampling dapat membuat struktur yang secara visual meyakinkan tetapi **tidak ada** pada sinyal kontinu. Pre-filter mereduksi komponen di luar pita yang dapat dicuplik; interpolasi setelah sampling tidak otomatis membalikkan aliasing.
3. Kuantisasi tidak setara dengan noise acak untuk semua masukan. Varians $\Delta^2/12$ memerlukan asumsi eksplisit; saturasi menghilangkan informasi.
4. Noise sensor bergantung pada sinyal. Varians temporal pada akuisisi berulang lebih relevan untuk analisis noise dibanding varians spasial satu frame tanpa pengendalian struktur.
5. RGB, luma, dan luminans linear tidak identik. Penelitian yang bergantung pada nilai warna memerlukan kontrol pipeline dan kalibrasi.
6. Perubahan representasi yang sederhana dapat mengubah perilaku model ML, tetapi temuan pada dataset simulasi/bawaan tidak boleh langsung ditafsirkan sebagai bukti performa perangkat di lapangan.

**Koneksi berikutnya:** histogram dan transformasi intensitas memperdalam distribusi sinyal; filtering dan domain frekuensi memperdalam PSF/OTF; inverse problems membahas rekonstruksi di bawah noise dan informasi yang tidak lengkap; pengukuran ilmiah membahas propagasi ketidakpastian dan validasi metrologi.

## Referensi utama dan catatan provenance

**Bacaan dasar dan metodologi:**

- R. C. Gonzalez dan R. E. Woods, *Digital Image Processing*, edisi ke-4, Pearson, 2018 — representasi, sampling, kuantisasi, dan transformasi citra.
- A. K. Jain, *Fundamentals of Digital Image Processing*, Prentice Hall, 1989 — hubungan sampling, transformasi, dan sistem linear.
- J. W. Goodman, *Introduction to Fourier Optics*, edisi ke-4, W. H. Freeman, 2017 — PSF, OTF, dan pembentukan citra koheren/inkoheren.
- European Machine Vision Association, [EMVA 1288 — standard for characterization of image sensors and cameras](https://www.emva.org/standards-technology/emva-1288/) — kerangka karakterisasi sensor.
- ICC Color Encoding Registry, [sRGB / IEC 61966-2-1](https://registry.color.org/rgb-registry/srgb) — definisi transfer function dan primaries.
- [scikit-image `transform.resize` documentation](https://scikit-image.org/docs/stable/api/skimage.transform.html#skimage.transform.resize) — *anti_aliasing*, *preserve_range*, interpolasi.
- [scikit-learn Digits dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html) — definisi dataset mini-studi kasus.

**Provenance eksperimen:** (1) `skimage.data.camera` dan `skimage.data.astronaut` adalah citra contoh bawaan library, bukan hasil kamera yang diketahui kalibrasinya; (2) pola spasial, kurva kuantisasi, flat-field Poisson-Gaussian, dan contoh kalibrasi adalah **data sintetis**; (3) dataset `load_digits` bawaan scikit-learn digunakan hanya untuk ilustrasi dampak representasi terhadap klasifikasi; (4) seed generator 2026; (5) seluruh library diperiksa di awal notebook pada lingkungan yang menjalankannya.

**Integritas ilmiah:** angka dari simulasi adalah uji mekanisme model, bukan pengukuran performa sensor fisik. Tidak ada klaim validasi instrumen, inferensi kausal pada kamera, atau keunggulan model tanpa uji independen.